# LoRA 训练 - Google Colab 版

基于 Qwen2.5-3B-Instruct 微调，角色扮演对话生成

**运行环境**: Google Colab (T4 GPU, 15GB RAM)**

## 1. 环境准备

In [ ]:
# 安装依赖
!pip install -q transformers datasets peft bitsandbytes accelerate safetensors scipy scikit-learn

In [ ]:
import torch
print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"显存: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. 克隆项目并准备数据

In [ ]:
# 克隆项目（如果有 Git 仓库）
# !git clone https://your-repo-url.git
# %cd your-project

# 或者直接在 Colab 中上传项目文件

from google.colab import files
import os

# 创建项目目录
os.makedirs("processed", exist_ok=True)
os.makedirs("configs", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

## 3. 数据准备

In [ ]:
from datasets import load_dataset, Dataset
import json
from sklearn.model_selection import train_test_split

# 加载 PIPPA 数据集
print("=== 加载数据集 ===")
dataset = load_dataset("KaraKaraWitch/PIPPA-ShareGPT-formatted", split="train")
print(f"原始数据: {len(dataset)} 条")

# 数据清洗
print("\n=== 数据清洗 ===")
cleaned_data = []
for item in dataset:
    if not item.get("conversations"):
        continue
    conversations = item["conversations"]
    if not any(m.get("from") == "system" for m in conversations):
        continue
    msgs = [m for m in conversations if m.get("from") in ["human", "gpt"]]
    if len(msgs) < 2:
        continue
    total_len = sum(len(m.get("value", "")) for m in conversations)
    if total_len > 8000:
        continue
    cleaned_data.append(item)

print(f"清洗后数据: {len(cleaned_data)} 条")

# 划分训练集和验证集
train_data, val_data = train_test_split(cleaned_data, test_size=0.1, random_state=42)
print(f"训练集: {len(train_data)} 条")
print(f"验证集: {len(val_data)} 条")

# 保存到本地
print("\n=== 保存数据 ===")
for split_name, data in [("train", train_data), ("val", val_data)]:
    with open(f"processed/{split_name}.jsonl", "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    print(f"已保存: processed/{split_name}.jsonl ({len(data)} 条)")

## 4. 配置与模型加载

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType, BitsAndBytesConfig
import torch

# 模型配置
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
MAX_SEQ_LENGTH = 2048

# LoRA 配置
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05

print("=== 加载分词器 ===")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    padding_side="right"
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("=== 加载模型 (4-bit 量化) ===")
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    quantization_config=quantization_config,
    device_map="auto",
)
print(f"模型加载完成，参数量: {model.num_parameters() / 1e6:.1f}M")

## 5. 应用 LoRA

In [ ]:
print("=== 应用 LoRA ===")
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    inference_mode=False,
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 6. 数据预处理

In [ ]:
from scripts.data_loader import load_local_dataset, format_conversation, tokenize_function
import sys
sys.path.insert(0, ".")

print("=== 加载本地数据 ===")
train_data, val_data = load_local_dataset("./processed")
print(f"训练集: {len(train_data)} 条")
print(f"验证集: {len(val_data)} 条")

from datasets import Dataset

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

print("\n=== 数据预处理 ===")
train_dataset = train_dataset.map(
    lambda x: format_conversation(x, tokenizer),
    remove_columns=train_dataset.column_names,
    desc="格式化训练集"
)
val_dataset = val_dataset.map(
    lambda x: format_conversation(x, tokenizer),
    remove_columns=val_dataset.column_names,
    desc="格式化验证集"
)
train_dataset = train_dataset.map(
    lambda x: tokenize_function(x, tokenizer, MAX_SEQ_LENGTH),
    remove_columns=["text"],
    desc="Tokenize 训练集"
)
val_dataset = val_dataset.map(
    lambda x: tokenize_function(x, tokenizer, MAX_SEQ_LENGTH),
    remove_columns=["text"],
    desc="Tokenize 验证集"
)

print(f"预处理完成: 训练集 {len(train_dataset)} 条, 验证集 {len(val_dataset)} 条")

## 7. 开始训练

In [ ]:
from transformers import DataCollatorForLanguageModeling

print("=== 创建训练器 ===")

training_args = TrainingArguments(
    output_dir="./output/lora_roleplay",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=100,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_steps=200,
    eval_steps=200,
    eval_strategy="steps",
    save_total_limit=3,
    fp16=True,
    optim="paged_adamw_8bit",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

print("=== 开始训练 ===")
trainer.train()

## 8. 保存模型

In [ ]:
print("=== 保存模型 ===")
final_dir = "./output/lora_roleplay/final_model"
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)
print(f"模型已保存到: {final_dir}")

# 下载模型文件
import shutil
shutil.make_archive("lora_model", "zip", final_dir)
files.download("lora_model.zip")